In [ ]:
# https://musicbrainz.org/doc/MusicBrainz_API
#
# See also:
# - https://musicbrainz.org/ws/2/artist?query=artist:%22Nathan%20Ross%22

In [ ]:
# Uncomment if needed
# %pip install requests pandas python-dotenv rich

In [ ]:
# Define Functions: Structure Normalization

def get_releases_and_recordings_for_artist_relations_data(l_relations_data: list):
    """
    :param l_relations_data: e.g., returned by mb_get(f"artist/{artist_id}", ...)["relations"]
    :return: app-normalized lists of releases and recordings
    """

    l_releases = []
    l_recordings = []

    for l_relation in l_relations_data:

        l_mb_type = l_relation["type"]
        if l_mb_type != "instrument":
            assert l_mb_type == "discogs"
            continue

        l_has_release = "release" in l_relation
        l_has_recording = "recording" in l_relation
        assert l_has_release != l_has_recording # always one or the other; never both

        l_credit = l_relation["source-credit"]
        l_roles = l_relation["attributes"]

        l_direction = l_relation["direction"]
        assert l_direction == "forward"

        l_begin = l_relation["begin"]
        l_end = l_relation["end"]

        if l_has_release:
            l_release = l_relation["release"]
            l_releases.append({
                "date": l_release["date"],
                "country": l_release["country"],
                "title": l_release["title"],
                "id": l_release["id"],
                "credit": l_credit,
                "roles": l_roles,
            })
            continue

        if l_has_recording:
            l_recording = l_relation["recording"]
            l_recordings.append({
                "begin": l_begin,
                "end": l_end,
                "title": l_recording["title"],
                "id": l_recording["id"],
                "credit": l_credit,
                "roles": l_roles,
            })
            continue

        assert False    # should never happen

    return l_releases, l_recordings


def urls_for_relations_data(l_relations_data: list) -> set[str]:
    l_urls = set()
    for l_relation in l_relations_data:
        pprint(l_relation)
        if l_relation.get("target-type") == "url":
            l_urls.add(l_relation["url"]["resource"])

    return l_urls


In [ ]:
# Define Functions: Load and Parse Data using MusicBrainz APIs

from pprint import pprint
import requests

# retrieve info from MusixBrainz; see also:
# - https://musicbrainz.org/relationships
# - https://musicbrainz.org/doc/MusicBrainz_Entity
def mb_get(l_endpoint: str, **l_params):
    l_params["fmt"] = "json"

    r = requests.get(
        f"https://musicbrainz.org/ws/2/{l_endpoint}",
        params=l_params,
        headers={
            "User-Agent": "NathanRossQueryAgent/0.1 (noodnik2@gmail.com)"
        },
        timeout=30,
    )

    r.raise_for_status()
    return r.json()


def player_urls_for_release(l_release_mbid: str) -> set[str]:
    """
    Returns every player URL known to MusicBrainz for this release.
    """

    # URLs attached to releases
    l_release_data = mb_get(
        f"release/{l_release_mbid}",
        # inc="isrcs", # see https://musicbrainz.org/doc/MusicBrainz_API#isrc
        inc="url-rels+recordings+isrcs",
        # inc="url-rels+isrcs",
        # inc="url-rels",
        limit=1, # TODO: remove this temporary change to avoid throttling
    )

    return urls_for_relations_data(l_release_data.get("relations", []))


def player_urls_for_recording(l_recording_mbid: str) -> set[str]:
    """
    Returns every player URL known to MusicBrainz for
    this recording or any release containing it.
    """

    l_recording_data = mb_get(
        f"recording/{l_recording_mbid}",
        # inc="releases+isrcs",
        inc="url-rels+releases+isrcs",
        # inc="url-rels+releases",
        limit=1, # TODO: remove this temporary change to avoid throttling
    )

    l_urls = set()

    # URLs attached directly to the recording
    l_urls = l_urls.union(urls_for_relations_data(l_recording_data.get("relations", [])))

    # URLs attached to releases
    for l_release in l_recording_data.get("releases", []):
        l_urls = l_urls.union(player_urls_for_release(l_release["id"]))

    return l_urls


In [ ]:
# Find Nathan Ross in online MusicBrainz Database

loaded_artist_info = mb_get(
    "artist",
    query='artist:"Nathan Ross"',
    limit=50,
)

In [ ]:
# Extract the single `artist_id` expected in MusicBrainz response

artists = loaded_artist_info["artists"]
assert len(artists) == 1
artist = artists[0]
artist_id = artist["id"]

In [ ]:
# Load the MusicBrainz `artist` Entity for `artist_id`

artist = mb_get(
    f"artist/{artist_id}",
    inc="artist-rels+recording-rels+release-rels+url-rels"
)

In [ ]:
# Retrieve normalized `releases` and `recordings` for returned MusicBrainz `artist`

releases, recordings = get_releases_and_recordings_for_artist_relations_data(
    artist["relations"]
)

In [ ]:
# Create Releases Dataframe

import pandas as pd

releases_pd = pd.DataFrame(out_releases)
releases_pd.head()

In [ ]:
# Create Recordings Dataframe

import pandas as pd

recordings_pd = pd.DataFrame(out_recordings)
recordings_pd.head()

In [ ]:
# To retrieve a song using Apple Music, try using the `isrc` attribute, e.g.:
# - https://music.apple.com/v1/catalog/us/songs?filter[isrc]=ITCF41902278
# - https://isrc.fm/isrc/USCA20300567


# Recordings
recording_mbid="298da2a1-c0dc-46ad-a593-2977f5a8a883"
# recording_mbid="601a8791-3e90-49ea-884a-0b49bd5a38fd"
urls = player_urls_for_recording(recording_mbid)

# Releases
release_mbid="8cf524c5-e9ee-45b0-bb79-fd176424f9a4"
# urls = player_urls_for_release(release_mbid)

pprint(urls)